# 🥇 Gold Layer — Retention Cohorts
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/gold/`, builds monthly cohort retention tables segmented overall, by acquisition channel, and by country, and writes to `medallion/gold/`.

| Gold Table | Gold Tables | Key Transformations |
|---|---|---|
| `gold_customer_orders` | `gold_retention_cohorts`, `gold_retention_cohorts_channel`, `gold_retention_cohorts_country` | Assign cohort month & period number, compute cohort sizes & active customers, calculate retention rates & avg revenue, segment by channel and country (SG/MY) |

## 0. Setup & Paths

In [29]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Retention Cohorts (`gold_customer_orders` → `gold_retention_cohorts*`)

### Step 1 — Load `gold_customer_orders` & prepare order-level data

In [30]:
# Load gold_customer_orders
co = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')

# Prepare order-level data
co['processed_at'] = pd.to_datetime(co['processed_at'], utc=True)
co = co.dropna(subset=['customer_id', 'processed_at'])

# Exclude Bulk Import — migrated historical orders, incomplete history
co = co[co['channel'] != 'Bulk Import'].copy()
print(f"Orders after excluding Bulk Import: {len(co):,}")

Orders after excluding Bulk Import: 27,901


### Step 2 — Assign cohort month and period number

In [31]:
# Assign cohort month and period number
co['order_quarter']   = co['processed_at'].dt.tz_localize(None).dt.to_period('Q')
cohort_map            = co.groupby('customer_id')['order_quarter'].min().rename('cohort_quarter')
co                    = co.join(cohort_map, on='customer_id')
co['period_number']   = (co['order_quarter'] - co['cohort_quarter']).apply(lambda x: x.n)

### Step 3 — Build overall cohort retention table

In [32]:
# Build overall cohort retention table
# Cohort size = unique customers in period 0
cohort_size = (co[co['period_number'] == 0]
               .groupby('cohort_quarter')['customer_id']
               .nunique()
               .rename('cohort_size'))

# Active customers per cohort per period
cohort_activity = (co.groupby(['cohort_quarter', 'period_number'])['customer_id']
                     .nunique()
                     .reset_index()
                     .rename(columns={'customer_id': 'active_customers'}))

# Merge and calculate retention rate
cohort_retention = cohort_activity.merge(cohort_size, on='cohort_quarter')
cohort_retention['retention_rate'] = (
    cohort_retention['active_customers'] / cohort_retention['cohort_size']
).round(4)

# Add avg revenue per customer per period
avg_revenue = (co.groupby(['cohort_quarter', 'period_number'])['price_total']
                 .mean()
                 .round(2)
                 .reset_index()
                 .rename(columns={'price_total': 'avg_revenue_per_customer'}))
cohort_retention = cohort_retention.merge(avg_revenue, on=['cohort_quarter', 'period_number'])

### Step 4 — Build channel-segmented cohort table

In [33]:
# Build channel-segmented cohort table
# Acquisition channel from first order
acquisition_channel = (co[co['order_sequence'] == 1][['customer_id', 'channel']]
                         .rename(columns={'channel': 'acquisition_channel'}))
co_channel = co.merge(acquisition_channel, on='customer_id', how='left')

channel_cohort_size = (co_channel[co_channel['period_number'] == 0]
                        .groupby(['cohort_quarter', 'acquisition_channel'])['customer_id']
                        .nunique()
                        .rename('cohort_size'))

channel_activity = (co_channel.groupby(['cohort_quarter', 'period_number', 'acquisition_channel'])
                               ['customer_id']
                               .nunique()
                               .reset_index()
                               .rename(columns={'customer_id': 'active_customers'}))

channel_cohort = channel_activity.merge(
    channel_cohort_size, on=['cohort_quarter', 'acquisition_channel'])
channel_cohort['retention_rate'] = (
    channel_cohort['active_customers'] / channel_cohort['cohort_size']
).round(4)

### Step 5 — Build country-segmented cohort table (SG & MY)

In [34]:
# Build country-segmented cohort table 
acquisition_country = (co[co['order_sequence'] == 1][['customer_id', 'country_code']]
                         .rename(columns={'country_code': 'acquisition_country'}))
co_country = co.merge(acquisition_country, on='customer_id', how='left')

# Keep SG and MY only for Q8
co_sg_my = co_country[co_country['acquisition_country'].isin(['sg', 'my'])].copy()

country_cohort_size = (co_sg_my[co_sg_my['period_number'] == 0]
                        .groupby(['cohort_quarter', 'acquisition_country'])['customer_id']
                        .nunique()
                        .rename('cohort_size'))

country_activity = (co_sg_my.groupby(['cohort_quarter', 'period_number', 'acquisition_country'])
                             ['customer_id']
                             .nunique()
                             .reset_index()
                             .rename(columns={'customer_id': 'active_customers'}))

country_cohort = country_activity.merge(
    country_cohort_size, on=['cohort_quarter', 'acquisition_country'])
country_cohort['retention_rate'] = (
    country_cohort['active_customers'] / country_cohort['cohort_size']
).round(4)

### Step 6 — Save all three cohort tables

In [35]:
# Save all three cohort tables
gold_retention_cohorts          = cohort_retention
gold_retention_cohorts_channel  = channel_cohort
gold_retention_cohorts_country  = country_cohort

gold_retention_cohorts.to_parquet(GOLD_DIR / 'gold_retention_cohorts.parquet', index=False)
gold_retention_cohorts_channel.to_parquet(GOLD_DIR / 'gold_retention_cohorts_channel.parquet', index=False)
gold_retention_cohorts_country.to_parquet(GOLD_DIR / 'gold_retention_cohorts_country.parquet', index=False)

print(f"gold_retention_cohorts:         {gold_retention_cohorts.shape[0]:,} rows × {gold_retention_cohorts.shape[1]} cols")
print(f"gold_retention_cohorts_channel: {gold_retention_cohorts_channel.shape[0]:,} rows × {gold_retention_cohorts_channel.shape[1]} cols")
print(f"gold_retention_cohorts_country: {gold_retention_cohorts_country.shape[0]:,} rows × {gold_retention_cohorts_country.shape[1]} cols")

# Quick sanity check — overall quarter retention rate
q1 = cohort_retention[cohort_retention['period_number'] == 1]
q3 = cohort_retention[cohort_retention['period_number'] == 3]
print(f"\nOverall quarter-1 retention rate: {(q1['active_customers'].sum() / q1['cohort_size'].sum()):.1%}")
print(f"Overall quarter-3 retention rate: {(q3['active_customers'].sum() / q3['cohort_size'].sum()):.1%}")

print("\n✅ Saved: all retention cohort tables")

gold_retention_cohorts:         326 rows × 6 cols
gold_retention_cohorts_channel: 812 rows × 6 cols
gold_retention_cohorts_country: 640 rows × 6 cols

Overall quarter-1 retention rate: 14.0%
Overall quarter-3 retention rate: 7.4%

✅ Saved: all retention cohort tables


#### Retention rates

Quarter-1 retention: 14.0% — 14% of customers placed another order within the first 3 months after acquisition. This is more meaningful than the 7.9% monthly figure since it captures customers who reorder at the natural 6-8 week product consumption cycle

Quarter-3 retention: 7.4% — drops to 7.4% by the 9-12 month mark, roughly half of Q1 retention

Q3 (Critical Window) — the drop from 14.0% (Q1) to 7.4% (Q3) confirms the steepest decline happens in the first quarter after acquisition. The 2-3 month product lifespan hypothesis holds — customers who don't reorder within their first tub's consumption window are much less likely to ever return.